In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'Scripts'))  # helper modules (run_config, obs, standardplots) live in LPNE/Scripts

from run_config import PROJECT  # project root (LPNE/), found relative to run_config.py - no hard-coded paths
import LPNE
import AREmulator
import run_config as rc
import statschecker as sc
import standardplots as sp
import longterm_insolation as li
import obs

import numpy as np
import numpy.random as npr
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, Model
import tensorflow.keras.backend as K
import keras as ke
from tensorflow.keras import regularizers
import optuna as opt
seed = 42

### Setup

In [ ]:
# config
DATASET = rc.DATASET        # '1461ka'
TRAIN_START = 'balanced'         # key of rc.STARTS: 'early', 'mid', 'late', 'balanced' (1461-579 ka, no test set) or 'default' (80:20 split)
INSOL = 'insol_la04'        # Laskar et al. (2004) 65N June-solstice insolation, computed on the fly by rc.load_data
END_KYR = 10000             # roll out to 10 Ma after present (the Laskar solution itself reaches +21 Ma)
N_FEATURES = rc.NFEATURES

PARAMS = {'conv filters': 30, 'conv kernel': 10, 'dense unit': 28, 
          'dropout rate': 0.3353763108415839, 'learning rate': 0.00038902232764492545, 
          'dwt gamma': 0.6506584806454648, 'dwt beta': 0.4637125670415251, 'dwt eps': 4.1664542731895874e-06, 
          'dwt levels': 2} # taken from optimize_model.ipynb, the 'mid' start optimization set

cfg = rc.make_config(DATASET, seed=seed, params=PARAMS, insol=INSOL, train_start=TRAIN_START)
params = cfg['params']
paths = rc.run_paths(cfg['run_tag'])

print("run tag :", cfg['run_tag'])
print("weights :", paths['weights'])
print("params  :", params)

In [ ]:
# load in and split data (the Laskar insolation column is computed here, not read from disk)
raw = rc.load_data(cfg)
splits = rc.split_data(raw, cfg)
print(rc.describe_split(splits))
testt, traint, valt = splits['test_time'], splits['train_time'], splits['validation_time']

# normalize data
scale, train, val, test = rc.fit_and_transform_splits(splits)

# chunk normalized data into windows
# segments: training can be in pieces (the 'balanced' start validates on the MPT); no window spans a gap
X_train, Y_train, Insol_train = LPNE.create_windows(train, cfg['window_size'], cfg['rollout_length'],
                                                   segments=splits['train_segments'])
X_val, Y_val, Insol_val = LPNE.create_windows(val, cfg['window_size'], cfg['rollout_length_validation'])

### Training

In [ ]:
LPNE.set_seeds(cfg['seed'])
brain = LPNE.CNN_model(cfg['window_size'], params=params, n_features=N_FEATURES)

emu = AREmulator.RolloutEmulator(brain,
                                rollout_length=cfg['rollout_length'],
                                vali_rollout_length=cfg['rollout_length_validation'],
                                window_size=cfg['window_size'],
                                n_features=N_FEATURES,
                                dwt_gamma=params['dwt gamma'],
                                dwt_beta=params['dwt beta'],
                                dwt_eps=params['dwt eps'],
                                dwt_levels=params['dwt levels'],
                                noise_level=cfg['noise_level'],
                                )
emu.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=params['learning rate'], weight_decay=1e-1, clipnorm=1.0))

dummy_X = tf.zeros((1, cfg['window_size'], N_FEATURES))
_ = emu(dummy_X)

# creates runs/<run_tag>/ and writes config.json with the scaler used for this data
rc.save_run(cfg, scale, splits)

check = sc.PhysicalAttractorCheckpoint(
    filepath = paths['weights'],
    acf_threshold = cfg['acf_threshold']
)
logger = sc.WeightStatsLogger()

history = emu.fit(
    [X_train, Insol_train],
    Y_train,
    validation_data=([X_val, Insol_val], Y_val),
    epochs=cfg['epochs'],
    callbacks=[check, logger],
    batch_size=cfg['batch_size'],
    validation_batch_size=cfg['batch_size'],
    shuffle=True,
)

### Long-term rollout (to 10 Ma after present)

In [ ]:
print(f"Loading best physical model for final evaluation from {paths['weights']} ...")
emu.load_weights(str(paths['weights']))
print(f"Best physical model found at epoch {check.best_epoch} with ACF threshold {cfg['acf_threshold']}.")

In [ ]:
# long-term forcing record: rows run oldest -> youngest, from the oldest age of the record (1461 ka BP) to
# END_KYR after present. Time is in kyr from present (negative = past), the Laskar convention.
# X, Y, Z are the GP mean where observations exist and NaN in the future: the rollout only reads the state of
# its seed window, and the insolation column (Laskar) at every step
years = np.arange(-cfg['max_age'], END_KYR + 1)
n_obs = splits['full'].shape[0]                       # rows with observations (1461 ka BP to present)
assert np.allclose(-splits['time'], years[:n_obs]), "observed ages must be the first rows of the long record"

long = np.full((years.size, N_FEATURES), np.nan)
long[:n_obs, :3] = splits['full'][:, :3]
long[:, 3] = li.solstice_insolation(years)
assert np.allclose(long[:n_obs, 3], splits['full'][:, 3]), "Laskar insolation must match the training forcing"
long_scaled = scale.transform(long)                   # NaN states stay NaN

# rolled out from the usual starting point (the oldest window, 1461 ka BP) to the end of the insolation record
w = cfg['window_size']
rollout_long = sp.rollout(emu, long_scaled, w)
print(f"rolled out {rollout_long.shape[0] - w} steps: {-years[0]:.0f} ka BP -> {years[-1]:.0f} kyr after present")

In [ ]:
# long-term emulator rollout with the GP mean ground truth underneath it (which ends at the present)
# edges of training+validation, in kyr from present: test | train (if there is a test set) and
# train/val | held out (if training stops before the present, e.g. the 'balanced' start)
tv = splits['trainval_time']
BOUNDS = []   # (time, label)
if len(testt):
    BOUNDS.append((-(testt.min() + tv.max()) / 2, 'test  |  train / validation'))
if len(splits['holdout_time']):
    BOUNDS.append((-(tv.min() + splits['holdout_time'].max()) / 2, 'train / validation  |  held out'))
COLORS = [["#D6195E","#FFB600"],["#1E88E5","#6519C5"],["#05EFC5","#8DB1A7"]]   # sp.plot_rollout's palette
truth = long_scaled[:n_obs]

with plt.rc_context({'font.size': 16}):
    fig, axs = plt.subplots(4, 1, figsize=(22, 15), sharex=True)

    for i in range(3):
        axs[i].plot(years[:n_obs], truth[:, i], color=COLORS[i][0], linestyle='dashed', lw=2, alpha=0.6,
                    label="real scaled data", zorder=1)
        axs[i].plot(years, rollout_long[:, i], color=COLORS[i][1], lw=1.5, label="emulator data", zorder=2)
        # error over the observed period only, seed window excluded
        mse = mean_squared_error(truth[w:, i], rollout_long[w:n_obs, i])
        axs[i].set_title(r"$\mathcal{L}_{L_2}$" + f" (observed period) = {mse:.3f}")
        axs[i].set_ylabel(sp.MAPPINGS[i])
        axs[i].legend(loc='upper left', fontsize=14)
        axs[i].grid(True, alpha=0.3)

    # insolation: given, not predicted
    axs[3].plot(years, long_scaled[:, 3], color='coral', lw=1)
    axs[3].set_title("Insolation forcing (Laskar 2004, 65N June solstice)")
    axs[3].set_ylabel("Insolation (scaled)")
    axs[3].grid(True, alpha=0.3)

    for ax in axs:
        for t_b, _ in BOUNDS:
            ax.axvline(t_b, color='black', linestyle=':', lw=2)
        ax.axvline(0, color='dimgray', linestyle=':', lw=2)
    for t_b, label in BOUNDS:
        axs[0].annotate(label, xy=(t_b, 1.0), xycoords=('data', 'axes fraction'),
                        xytext=(0, 6), textcoords='offset points', ha='center', va='bottom', fontsize=14)
    axs[0].annotate('present  |  future', xy=(0, 1.0), xycoords=('data', 'axes fraction'),
                    xytext=(0, 6), textcoords='offset points', ha='left', va='bottom', fontsize=14, color='dimgray')

    axs[-1].set_xlabel("time [kyr from present]  (negative = past)")
    plt.tight_layout()
    plt.show()

In [ ]:
# ---- Phase space: ground truth in two periods over the long-term emulator rollout ----
# Ground truth is the GP mean of the observations (the X, Y, Z columns of long_scaled), split into an early
# (1461-1200 ka) and a late (800-0 ka) Pleistocene period; the full 10 Ma rollout is drawn faintly behind it.
# Scaled units, as in the rollout plot above. X = ice mass, Y = CO2, Z = deep ocean temperature
PHASE_PERIODS = {   # label -> (oldest, youngest) age [ka], color
    '1461-1200 ka': ((1461, 1200), '#1E88E5'),
    '800-0 ka':     ((800, 0),     '#D6195E'),
}
ROLLOUT_ALPHA = 0.5                   # low, so the ground truth stays easy to see
ages_obs = -years[:n_obs]              # age [ka BP] of each observed row, oldest -> youngest
PAIRS = [(0, 1), (0, 2), (1, 2)]       # (x, y) state-variable indices per panel

with plt.rc_context({'font.size': 16}):
    fig, axs = plt.subplots(1, 3, figsize=(22, 7))

    for ax, (a, b) in zip(axs, PAIRS):
        ax.plot(rollout_long[1461:, a], rollout_long[1461:, b], color='dimgray', lw=0.8, alpha=ROLLOUT_ALPHA,
                label=f"Emulator rollout (1461 ka BP to +{END_KYR / 1000:.0f} Ma)", zorder=1)
        ax.scatter(rollout_long[1461-800, a], rollout_long[1461-800, b], color = 'black', label = 'Emulator @ 800 kaBP', edgecolor = 'black', s=60, zorder = 3)

        for label, ((oldest, youngest), color) in PHASE_PERIODS.items():
            seg = (ages_obs <= oldest) & (ages_obs >= youngest)
            ax.plot(truth[seg, a], truth[seg, b], color=color, lw=1.8, alpha=0.9,
                    label=f"ground truth, {label}BP", zorder=2)
            first = np.flatnonzero(seg)[0]    # oldest row of the period
            ax.scatter(truth[first, a], truth[first, b], color=color, edgecolor='black', s=60, zorder=3)

        ax.set_xlabel(sp.MAPPINGS[a])
        ax.set_ylabel(sp.MAPPINGS[b])
        ax.grid(True, alpha=0.3)

    handles, labels = axs[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=len(handles), fontsize=15, frameon=True,
               title='GP mean ground truth (dot = start of each period) over the long-term rollout')
    plt.tight_layout(rect=[0, 0.12, 1, 1])
    plt.show()